# 🚀 Hệ thống Quét CCCD/CMND ra Excel - Phiên bản Tốc Độ Bàn Thờ (Colab Pro-Tip)

Chào mừng bạn đến với phiên bản được **tối ưu hóa I/O và AI** chạy trên **Google Colab (GPU T4)**.
Thay vì đọc hàng ngàn ảnh trực tiếp từ Drive rất chậm chạp, hệ thống này sử dụng quy trình chuẩn: **Copy file .ZIP vào Colab -> Giải nén tốc độ cao -> Ép xung bằng `onnxruntime-gpu` -> Nén output và trả về Drive**.
Tốc độ quét có thể đạt tới **gấp 10-20 lần** so với phiên bản đọc file rời.

--- 
### 📌 QUY TRÌNH THỰC HIỆN:
1. Trên máy tính của bạn, hãy **nén tất cả ảnh CCCD thành 1 file .zip** (ví dụ: `Anh_CCCD.zip`).
2. Tải file `Anh_CCCD.zip` đó lên **Google Drive**.
3. Chạy lần lượt các bước từ 1 đến 5 bên dưới bằng cách bấm nút Play (Tam giác).


In [ ]:
# @title ⚙️ Kiểm tra cấu hình GPU T4 (Tùy chọn)
# Ô này giúp kiểm tra xem bạn đã bật GPU T4 thành công chưa. Nếu thấy báo lỗi, hãy vào menu Runtime > Change runtime type > Chọn T4 GPU.
!nvidia-smi

In [ ]:
# @title 📂 BƯỚC 1: Kết nối Google Drive
# Cấp quyền để Colab lấy file .zip ảnh từ Drive của bạn
from google.colab import drive
import os

print("Đang yêu cầu quyền truy cập Google Drive...")
drive.mount('/content/drive')
print("✅ Kết nối Google Drive thành công!")

In [ ]:
# @title 📦 BƯỚC 2: Tải Mã Nguồn & Tối ưu hóa Thư viện GPU
# @markdown Quá trình này sẽ cài đặt lõi xử lý ảnh tốc độ cao và `onnxruntime-gpu`. Mất khoảng 1-2 phút.

import os
from IPython.display import clear_output

print("1/4. Đang tải thư viện hệ thống...")
!apt-get update -qq
!apt-get install -y libzbar0 libgl1-mesa-glx > /dev/null

print("2/4. Đang tải mã nguồn...")
%cd /content
if not os.path.exists('cccd-qr-excel'):
    !git clone https://github.com/vonguyendang/cccd-qr-excel.git
else:
    %cd cccd-qr-excel
    !git pull
    %cd ..

print("3/4. Đang cài đặt công cụ ép xung GPU (onnxruntime-gpu) & AI...")
%cd /content/cccd-qr-excel
!sed -i '/numpy<2.0.0/d' wizard/requirements.txt
!sed -i '/onnxruntime/d' deepdoc_vietocr/requirements.txt
!pip install pyzbar opencv-contrib-python-headless==4.10.0.84 openpyxl pillow-heif zxing-cpp==3.0.0 > /dev/null
!pip install torch torchvision vietocr pycryptodomex setuptools pdfplumber onnxruntime-gpu > /dev/null
!pip install "numpy<2.0.0" > /dev/null

clear_output()
print("✅ Môi trường GPU T4 đã sẵn sàng ở mức năng lượng tối đa!")

In [ ]:
# @title 📥 BƯỚC 3: Giải nén file ảnh siêu tốc (Tránh I/O Bottleneck)
# @markdown Nhập đường dẫn file **.ZIP** chứa ảnh CCCD trên Google Drive của bạn.
# @markdown Ví dụ: `/content/drive/MyDrive/Anh_CCCD.zip`
file_zip_tren_drive = "/content/drive/MyDrive/CCCD_Test.zip" # @param {type:"string"}

import os

if not os.path.exists(file_zip_tren_drive):
    print(f"❌ Lỗi: Không tìm thấy file '{file_zip_tren_drive}'!")
    print("Hãy bấm vào tab Files bên trái, tìm file zip và copy lại đường dẫn.")
else:
    print("1. Đang copy file zip vào ổ cứng SSD nội bộ của Colab (Tốc độ cực nhanh)...")
    !cp "{file_zip_tren_drive}" /content/images.zip
    print("2. Đang giải nén ảnh...")
    !rm -rf /content/Anh_CCCD
    !mkdir -p /content/Anh_CCCD
    !unzip -q -j /content/images.zip -d /content/Anh_CCCD
    print("✅ Giải nén hoàn tất! Đã sẵn sàng đưa vào GPU quét.")

In [ ]:
# @title 🚀 BƯỚC 4: Kích hoạt Quét AI
# @markdown Tiến trình sẽ đọc các ảnh trong thư mục vừa giải nén `/content/Anh_CCCD`.

import os
if not os.path.exists("/content/Anh_CCCD") or len(os.listdir("/content/Anh_CCCD")) == 0:
    print("❌ Lỗi: Thư mục chứa ảnh bị trống. Hãy chạy lại Bước 3 để giải nén ảnh.")
else:
    print("Đang nạp AI Models vào VRAM...")
    %cd /content/cccd-qr-excel/wizard
    
    # Chúng ta bypass input bằng pipe
    !echo "/content/Anh_CCCD" | python3 main.py
    
    print("\n=========================================")
    print("✅ QUÉT HOÀN TẤT!")
    print("Chuyển qua Bước 5 để đóng gói kết quả trả về Google Drive.")
    print("=========================================")

In [ ]:
# @title 📦 BƯỚC 5: Đóng gói và Copy kết quả về Google Drive
# @markdown Kết quả xuất ra sẽ được nén lại thành 1 file ZIP (bao gồm Excel và các ảnh đã đổi tên) và đẩy về Drive để bạn tải về.
thu_muc_luu_tren_drive = "/content/drive/MyDrive/KetQua_CCCD" # @param {type:"string"}

import shutil
import os

export_dir = "/content/cccd-qr-excel/wizard/exports"

if os.path.exists(export_dir) and len(os.listdir(export_dir)) > 0:
    if not os.path.exists(thu_muc_luu_tren_drive):
        os.makedirs(thu_muc_luu_tren_drive)
        
    subdirs = [os.path.join(export_dir, d) for d in os.listdir(export_dir) if os.path.isdir(os.path.join(export_dir, d))]
    if subdirs:
        latest_subdir = max(subdirs, key=os.path.getmtime)
        folder_name = os.path.basename(latest_subdir)
        
        # Nén thư mục kết quả thành ZIP để copy về Drive siêu nhanh
        zip_path = f"/content/{folder_name}"
        print("Đang đóng gói file Excel và các file ảnh xuất ra...")
        shutil.make_archive(zip_path, 'zip', latest_subdir)
        
        dest_path = os.path.join(thu_muc_luu_tren_drive, f"{folder_name}.zip")
        print(f"Đang đẩy file về Google Drive của bạn: {dest_path}")
        shutil.copy2(f"{zip_path}.zip", dest_path)
        print("\n✅ THÀNH CÔNG! Hãy mở Google Drive của bạn, tải file Zip mới nhất về máy và xem thành quả nhé!")
    else:
        print("❌ Không tìm thấy kết quả để lưu.")
else:
    print("❌ Không tìm thấy dữ liệu xuất ra. Có thể bạn chưa chạy Bước 4 thành công.")